# Optimizers & Learning Rate Schedules
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> SGD with Momentum, Nesterov, AdaGrad, RMSprop, Adam, AdamW (decoupled weight decay), and modern Learning Rate Schedules.

---

### 📖 Conceptual Foundations

#### The Optimization Challenge in Deep Learning

Deep network loss landscapes are high-dimensional, highly non-convex surfaces filled with saddle points, ill-conditioned ravines (where curvature is much steeper in some directions than others), and noisy stochastic gradients. Modern optimizers incorporate **momentum** to accelerate through ravines and **adaptive learning rates** to scale updates per parameter according to historical gradient magnitudes.


#### Evolution of First-Order Optimizers

| Optimizer | Update Equations | Key Mechanism | Ideal Use Case |
| --- | --- | --- | --- |
| SGD + Momentum | $v_t = \beta v_{t-1} + (1-\beta)g_t$
$\theta_t = \theta_{t-1} - \alpha v_t$ | Exponential moving average of gradients acts like a heavy ball rolling down a hill; dampens oscillations in steep ravines. | Computer Vision (ResNets, ConvNets), often achieves superior generalization when tuned. |
| Nesterov Accelerated Gradient (NAG) | $v_t = \beta v_{t-1} + \alpha \nabla f(\theta_{t-1} - \beta v_{t-1})$
$\theta_t = \theta_{t-1} + v_t$ | Computes gradient at look-ahead position; applies 'brakes' before overshooting valleys. | High-curvature optimization tasks. |
| AdaGrad (2011) | $s_t = s_{t-1} + g_t \odot g_t$
$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{s_t} + \epsilon} g_t$ | Divides learning rate by square root of sum of past squared gradients. Frequently decaying step size. | Sparse data (text/NLP word embeddings); stalls prematurely on deep networks. |
| RMSprop (Hinton, 2012) | $s_t = \beta s_{t-1} + (1-\beta)g_t^2$
$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{s_t} + \epsilon} g_t$ | Replaces infinite accumulation with exponentially decaying average of squared gradients. | Recurrent neural networks (RNNs), non-stationary problems. |
| Adam (Kingma & Ba, 2014) | $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$
$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$
$\hat{m}_t = m_t / (1-\beta_1^t), \; \hat{v}_t = v_t / (1-\beta_2^t)$ | Combines 1st moment (momentum) and 2nd moment (adaptive variance) with bias correction for zero initialization. | Default baseline across deep learning, fast early convergence. |
| AdamW (Loshchilov & Hutter, 2017) | Decouples $L_2$ weight decay $\lambda \theta$ directly from the adaptive gradient updates. | Fixes incorrect interaction between $L_2$ penalty and adaptive scale $\sqrt{v_t}$ in Adam. | Standard default for Transformers (BERT, GPT, Llama, ViT). |


#### Why AdamW Replaced Adam + L2 Regularization

In standard Adam with $L_2$ regularization, the weight penalty $\lambda \theta$ is added to the gradient $g_t$. The adaptive update then divides this term by $\sqrt{v_t} + \epsilon$. As a result, weights with large historical gradients receive *less* weight decay than weights with small gradients! **AdamW fixes this** by applying weight decay directly to $\theta$ outside the gradient division:

$$\theta_t = \theta_{t-1} - \alpha \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_{t-1} \right)$$


#### Learning Rate Schedules & Warmup

The learning rate is the single most important hyperparameter in deep learning:

1. **Linear Warmup:** Start from near zero for the first $K$ steps, ramping up to $\alpha_{max}$. Prevents unstable gradient updates when initial moments ($m_t, v_t$) are noisy.
2. **Cosine Annealing:** Smoothly decays $\alpha$ following a half-cosine curve down to a minimum $\alpha_{min}$, allowing gentle convergence into flat minima.
3. **OneCycle Policy (Smith, 2018):** Increases learning rate while decreasing momentum, then decreases learning rate to achieve rapid super-convergence.



In [ ]:
# %matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

class AdamW:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=1e-2):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0
        
    def step(self, grads):
        self.t += 1
        for i in range(len(self.params)):
            p = self.params[i]
            g = grads[i]
            
            # Decoupled weight decay update
            p -= self.lr * self.weight_decay * p
            
            # 1st and 2nd moment updates
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g ** 2)
            
            # Bias corrections
            m_hat = self.m[i] / (1.0 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1.0 - self.beta2 ** self.t)
            
            # Parameter update
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Test on 2D Rosenbrock (banana) function: f(x, y) = (1-x)^2 + 100(y - x^2)^2
theta = np.array([-1.5, 2.0])
opt = AdamW([theta], lr=0.05, weight_decay=1e-4)
trajectory = [theta.copy()]

for _ in range(300):
    x, y = theta[0], theta[1]
    grad_x = -2 * (1 - x) - 400 * x * (y - x**2)
    grad_y = 200 * (y - x**2)
    opt.step([np.array([grad_x, grad_y])])
    trajectory.append(theta.copy())

traj = np.array(trajectory)
print(f'Optimized Point: x={traj[-1, 0]:.4f}, y={traj[-1, 1]:.4f} (Global min at 1, 1)')

plt.figure(figsize=(6, 4))
plt.plot(traj[:, 0], traj[:, 1], 'ro-', markersize=2, label='AdamW Path')
plt.plot(1, 1, 'g*', markersize=12, label='Target (1,1)')
plt.title('AdamW Optimizing Rosenbrock Valley')
plt.xlabel('x'); plt.ylabel('y'); plt.grid(True); plt.legend()
plt.show()


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

